# Kaggriculture — Test & Submit Notebook

This notebook has two phases, run in order:

1. **Test** (Phases 0–5) — sanity-check that `main.py` runs, watch it play,
   then batch-test it for a real signal instead of trusting a single game.
2. **Submit** (Phases 6–8) — push it to the actual Kaggle competition and
   check on it once it's live.

**Before you start:** this assumes `main.py` sits in the same folder as this
notebook, and that `main.py` defines a top-level function named exactly
`agent(obs)` — that's the entry point Kaggle's own runtime looks for, so it
has to be named that, not `my_agent` or anything else.

Run cells top to bottom. Don't skip to Phase 6 until Phase 5 passes cleanly —
each phase is there because it catches a different class of bug, and they
build on each other.

## Phase 0 — Environment sanity check

Before testing any game logic, confirm the notebook kernel can actually see
`kaggle-environments` and that `main.py` is where the kernel expects it. This
matters more than it sounds like in a WinPython setup specifically, since
WPy64 is a self-contained distribution — if you have another Python install
on the machine, `pip install` can silently go to the wrong one. If the import
below fails, open the **WinPython Command Prompt** that ships with your WPy64
folder (that's the one guaranteed to point at the right `pip`) and run
`pip install -U kaggle-environments` from there, then restart this kernel.

In [ ]:
import os
import sys

print("Python executable:", sys.executable)

try:
    import kaggle_environments
    print("kaggle_environments version:", kaggle_environments.__version__)
    print("Loaded from:", kaggle_environments.__file__)
except ImportError as e:
    raise SystemExit(
        "kaggle_environments is not visible to this kernel.\n"
        "Install it from the WinPython Command Prompt (not a random other Python):\n"
        "    pip install -U kaggle-environments\n"
        f"Original error: {e}"
    )

cwd = os.getcwd()
print("\nNotebook working directory:", cwd)

main_path = os.path.join(cwd, "main.py")
if os.path.exists(main_path):
    print("Found main.py at:", main_path)
else:
    raise SystemExit(
        f"main.py was not found in {cwd}.\n"
        "Either move it next to this notebook, or set main_path below to the full path."
    )


**What a failure here means:** an `ImportError` means the package
isn't installed *for this kernel* — reinstalling in a random terminal window
won't fix it if that terminal points at a different Python. A missing
`main.py` almost always means the notebook's working directory isn't what
you assumed; the printed `cwd` above tells you exactly where the kernel is
looking, which is usually faster than guessing.

## Phase 1 — Single-game smoke test

The first real test: does the agent finish a full game without erroring out?
We pass `"main.py"` as a **string**, not a Python object, on purpose — that
loads and runs the actual file you'll eventually submit, byte for byte, so
there's no chance this test and your real submission quietly drift apart
because you edited one and forgot the other.

`debug=True` is what makes exceptions inside your agent surface here instead
of getting swallowed and just showing up as a mysterious loss. We run it
against all three built-in opponents once each: `"pass"` is a floor you
should beat easily, `"random"` checks how your agent copes with genuinely
unpredictable interference, and `"starter"` is Kaggle's own deterministic
baseline — roughly the level a lot of early opponents on the ladder will sit
at, so it's the one worth caring about most.

In [ ]:
from kaggle_environments import make

for opponent in ["pass", "random", "starter"]:
    env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)
    env.run(["main.py", opponent])

    final = env.steps[-1]
    my_status = final[0].status
    my_reward = final[0].reward
    opp_reward = final[1].reward

    outcome = "WIN" if my_reward > opp_reward else ("LOSS" if my_reward < opp_reward else "TIE")
    print(f"vs {opponent:8s} | status={my_status:8s} | you={my_reward!s:>8} | opp={opp_reward!s:>8} | {outcome}")


**Reading this output:** `status` should read `DONE` for every row —
anything else (commonly `ERROR` or `INVALID`) means your agent threw an
exception or timed out mid-game, and that's worth fixing before anything
below this, since a broken agent makes every later test meaningless. Don't
read much into win/loss/tie yet from just three games; that's what the batch
test in Phase 3 is actually for. This step is purely "does it run.

## Phase 2 — Watch a replay

A pass/fail check won't catch the embarrassing bugs — your farmer standing
still for six turns because a condition never fires, watering the wrong
tile, a sell order you thought you queued but didn't. Those are much easier
to spot by watching a game than by reading code. Worth doing at least once
early on.

In [ ]:
env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)
env.run(["main.py", "starter"])
env.render(mode="ipython", width=1200, height=800)


**If the cell above rendered nothing, or WPy64's Jupyter froze/stayed blank:**
this is a known rough edge, not a sign anything is actually broken. A single
full-game replay renders to a self-contained HTML/JS bundle that comes out
to roughly 20 MB, and some local Jupyter frontends (older "classic"
notebook builds especially, which is what some WinPython distributions
ship) sandbox or choke on embedding that much inline HTML in one output
cell. It's a display problem, not a simulation problem \u2014 the game already
finished running by the time render is called.

The fix: skip the inline notebook renderer and write the same HTML to a
standalone file, then open that file directly in a normal browser (Chrome,
Edge, Firefox), completely outside Jupyter. It's the exact same visualizer,
it just isn't fighting the notebook's output sandbox anymore.

(One honest correction to something I said earlier: I don't have confirmed
evidence that Kaggle hosts a general page where you upload an arbitrary
*local* replay JSON to view it \u2014 I went looking and couldn't verify that
exists. What Kaggle definitely has is a replay viewer for real episodes
your agent plays once it's actually submitted and on the ladder, reachable
from the competition's Leaderboard/Submissions pages. For a local test game
that never touched Kaggle's servers, the file-based approach below is the
reliable option.)

In [ ]:
html = env.render(mode="html")
with open("replay_view.html", "w", encoding="utf-8") as f:
    f.write(html)

print(f"Saved replay_view.html ({len(html)/1e6:.1f} MB) to {os.getcwd()}")
print("Open it directly in a browser (double-click it, or drag it into Chrome/Edge/Firefox) \u2014 not through Jupyter.")


The raw JSON is also worth keeping around separately from the HTML file above — useful if you ever want to script your own analysis over a game (step through specific turns, pull out price history, etc.) rather than just watching it:

In [ ]:
import json

with open("replay.json", "w") as f:
    json.dump(env.toJSON(), f)

print("Saved replay.json —", os.path.getsize("replay.json"), "bytes")


## Phase 3 — Batch testing for a real signal

One game is noisy — weeds spawn randomly, and the market reacts to whatever
the opponent happens to do that particular game, so a single win or loss
doesn't tell you much. This runs a batch with a **different seed each game**
(reusing one seed for all fifty just tests luck, not your agent), and
aggregates the outcome so you get an actual win rate to trust.

In [ ]:
def run_batch(agent_path="main.py", opponent="starter", n=50, base_seed=0):
    results = {"win": 0, "loss": 0, "tie": 0, "error": 0}
    for i in range(n):
        env = make(
            "kaggriculture",
            configuration={"episodeSteps": 720, "seed": base_seed + i},
            debug=True,
        )
        env.run([agent_path, opponent])
        final = env.steps[-1]

        if any(s.status != "DONE" for s in final):
            results["error"] += 1
            continue

        r0, r1 = final[0].reward, final[1].reward
        if r0 > r1:
            results["win"] += 1
        elif r0 < r1:
            results["loss"] += 1
        else:
            results["tie"] += 1
    return results


N_GAMES = 50  # bump this up once you trust the loop; 50 is a reasonable start
results = run_batch(opponent="starter", n=N_GAMES)

played = N_GAMES - results["error"]
win_rate = results["win"] / played if played else 0.0

print(f"Against 'starter' over {N_GAMES} games:")
print(results)
if played:
    print(f"Win rate (excluding errors): {win_rate:.1%}")
if results["error"]:
    print(f"\n⚠️  {results['error']} game(s) errored out — check these before trusting the win rate.")


**Reading this:** any nonzero `error` count means some seeds are
hitting a bug your three-opponent smoke test in Phase 1 didn't happen to
trigger — worth tracking down before you submit, since Kaggle's own
validation check (Phase 5) will catch it anyway and burn a submission slot
doing so. For the win rate itself, there's no universal "good" number since
it depends entirely on the field, but comfortably beating `"starter"` is the
minimum bar before you'd want to spend a real submission on it.

## Phase 3.5 — Compare against your previous version (optional)

Once you've got more than one iteration of `main.py`, win-rate-against-your-
*own-last-version* is a cleaner signal than win-rate-against-`starter`, since
it tells you directly whether a change helped. Keep old versions around
(e.g. `main_v1.py`, `main_v2.py`) and battle them head to head:

In [ ]:
# Uncomment once you have a previous version saved alongside this notebook.

# old_results = run_batch(agent_path="main.py", opponent="main_v1.py", n=50)
# played = 50 - old_results["error"]
# print(f"New version vs previous version: {old_results}")
# if played:
#     print(f"Win rate vs previous version: {old_results['win']/played:.1%}")


## Phase 4 — Endgame liquidation check (optional but worth it)

Unsold inventory doesn't count toward your final score, and the market
punishes dumping everything in one turn. That's an edge case that won't show
up reliably in an averaged win rate across full 720-turn games, so it's
worth checking on its own. This starts a game already near the end of the
season with `episodeSteps` cut short, just to eyeball what your sell-off
logic does under time pressure — you'll likely want to tailor this once you
know your own agent's inventory-heavy scenarios.

In [ ]:
env = make("kaggriculture", configuration={"episodeSteps": 24, "seed": 1}, debug=True)
env.run(["main.py", "starter"])
env.render(mode="ipython", width=1200, height=800)


## Phase 5 — Self-play validation *(don't skip this one)*

When you upload a submission, Kaggle automatically runs a **Validation
Episode**: your agent playing against a copy of itself, purely to confirm it
runs without erroring. If that fails, the submission is marked `Error` and
never enters the rating pool — no retry, it just costs you one of your five
submissions for the day. Running this exact scenario yourself first is cheap
insurance, and self-play sometimes surfaces bugs single-opponent testing
won't, like both farmers behaving oddly around the same shared resource.

In [ ]:
env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)
env.run(["main.py", "main.py"])

final = env.steps[-1]
statuses = [s.status for s in final]
print("Self-play statuses:", statuses)

if all(s == "DONE" for s in statuses):
    print("✅ Self-play finished cleanly — matches what Kaggle's own validation episode checks.")
else:
    print("❌ Self-play did not finish cleanly — fix this before submitting, or your submission will error out.")


## Pre-submission checklist

Before running Phase 6 below, confirm:

- You've **joined the competition and accepted the rules** at
  `https://www.kaggle.com/competitions/kaggriculture` (this can't be
  scripted — it has to be done on the website).
- `main.py` defines a top-level `agent(obs)` function, and nothing else it
  depends on lives outside that file (or you're bundling a `.tar.gz` — see
  the note in Phase 6).
- Phase 5 (self-play) finished with `DONE` for both sides.
- Your Phase 3 batch win rate against `"starter"` is one you're actually
  happy to spend a submission on.
- Your agent makes no network calls and reads nothing from outside itself —
  the rules explicitly forbid any ingress or egress during an evaluated
  episode, so double-check nothing like a stray API call snuck in during
  development.

## Phase 6 — Confirm the Kaggle CLI is ready

Quick check that the `kaggle` CLI is installed and your API credentials are
working, before trying to submit anything for real. This should print a row
for the Kaggriculture competition if everything's wired up; an auth error
here means your token isn't in place yet (see `~/.kaggle/access_token`, or
run `kaggle auth login`).

**WPy64 note:** if you see `'kaggle' is not recognized as an internal or
external command`, that's a PATH issue, not a missing install -- pip put
`kaggle.exe` in WinPython's `Scripts` folder, but that folder isn't
necessarily on the PATH the notebook's shell commands use, the same class
of issue as the `kaggle_environments` import earlier in this notebook.
Rather than fight PATH, the cells below call the package as a module
through the exact interpreter this kernel is already using
(`sys.executable`), which sidesteps the problem entirely regardless of
PATH -- that's what `{sys.executable} -m kaggle` is doing instead of a
bare `kaggle`.

In [ ]:
import sys

# Confirms the package itself is installed for *this* interpreter --
# separate from the PATH issue, and worth ruling out first if the next
# cell still fails after this fix.
!{sys.executable} -m pip show kaggle | findstr /B "Name Version"

!{sys.executable} -m kaggle competitions list -s "kaggriculture"


## Phase 7 — Submit

This is the real thing — running this cell uses one of your **five daily
submission slots** on the actual competition. Only run it once you've been
through the checklist above.

The message string (`-m "..."`) is your own note to yourself for telling
submissions apart later on the Submissions page, so make it specific enough
to be useful in a week.

In [ ]:
SUBMISSION_MESSAGE = "Heuristic v1 — wheat loop + staged land + throttled selling"

!{sys.executable} -m kaggle competitions submit kaggriculture -f main.py -m "{SUBMISSION_MESSAGE}"


If your agent needs helper files or a saved model (e.g. weights from
tuning), bundle everything into a `tar.gz` with `main.py` at the root
instead of submitting the single file:

```python
# import tarfile
# with tarfile.open("submission.tar.gz", "w:gz") as tar:
#     tar.add("main.py")
#     tar.add("helper.py")
#     tar.add("model_weights.pkl")
```

```
!kaggle competitions submit kaggriculture -f submission.tar.gz -m "{SUBMISSION_MESSAGE}"
```

## Phase 8 — Monitor your submission

Check status right after submitting, then come back later — the Validation
Episode (self-play, error-check only) runs fairly quickly, but it takes time
on the ladder for enough rated episodes to accumulate before the rating
means much.

In [ ]:
!{sys.executable} -m kaggle competitions submissions kaggriculture


Once the output above shows your submission and gives you its
**submission ID**, plug it in below to see the episodes it's played:

In [ ]:
SUBMISSION_ID = "PASTE_YOUR_SUBMISSION_ID_HERE"

!{sys.executable} -m kaggle competitions episodes {SUBMISSION_ID}


And once you have an **episode ID** from that list, you can pull the
replay and your agent's logs for real matches you didn't simulate locally —
genuinely useful for spotting blind spots against strategies other
participants are actually running:

In [ ]:
EPISODE_ID = "PASTE_AN_EPISODE_ID_HERE"

!{sys.executable} -m kaggle competitions replay {EPISODE_ID}
!{sys.executable} -m kaggle competitions logs {EPISODE_ID} 0


Finally, the live leaderboard:

In [ ]:
!{sys.executable} -m kaggle competitions leaderboard kaggriculture -s


## Notes for next iteration

- Rename this version of `main.py` (e.g. copy it to `main_v1.py`) before you
  start changing it, so Phase 3.5 has something to compare the next version
  against.
- Remember only your **latest two submissions** are tracked for matchmaking
  and count toward your final score — the daily cap of five is room to
  experiment, not five equally "real" shots.
- Re-run Phases 0–5 on every new version before spending a submission on it.
  It's a few minutes of local testing against a slot you only get five of
  per day.